# Authoritative AAS validation

This notebook evaluates an injected `attack.py` against the competition-matched GPT-OSS and Gemma GGUF models. Use `jed-validate kaggle run path/to/attack.py` to stage and execute it on a private Kaggle GPU kernel.

In [ ]:
import gc
import importlib.util
import json
import os
import subprocess
import sys
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path

COMP_DIR = Path("/kaggle/input/competitions/ai-agent-security-multi-step-tool-attacks")
GPT_OSS_PATH = Path(
    "/kaggle/input/models/llkh0a/gpt-oss-20b-gguf/pytorch/default/1/gpt_oss/gpt-oss-20b-Q4_K_M.gguf"
)
GEMMA_PATH = Path(
    "/kaggle/input/models/llkh0a/gemma-4-26b-a4b-it-ud-q4-k-m-gguf/"
    "pytorch/default/1/gemma/gemma-4-26B-A4B-it-UD-Q4_K_M.gguf"
)
WORK_DIR = Path("/kaggle/working")
ARTIFACTS_DIR = WORK_DIR / "artifacts"
ATTACK_PATH = WORK_DIR / "attack.py"
ATTACK_SHA256 = "not-injected"
RUN_INTERACTIVE_INSPECTION = False

WORK_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
assert COMP_DIR.exists(), f"Missing competition SDK: {COMP_DIR}"
assert GPT_OSS_PATH.exists(), f"Missing GPT-OSS GGUF: {GPT_OSS_PATH}"
assert GEMMA_PATH.exists(), f"Missing Gemma GGUF: {GEMMA_PATH}"
gpu_check = subprocess.run(["nvidia-smi"], check=False, capture_output=True, text=True)
assert gpu_check.returncode == 0, "This notebook requires a Kaggle NVIDIA GPU accelerator"
print(gpu_check.stdout.splitlines()[0] if gpu_check.stdout else "NVIDIA GPU available")

sys.path.insert(0, str(COMP_DIR))
os.environ["PYTHONUTF8"] = "1"
os.environ["GPT_OSS_MODEL_PATH"] = str(GPT_OSS_PATH)
os.environ["GEMMA_MODEL_PATH"] = str(GEMMA_PATH)

from kaggle_evaluation.jed_attack_134815 import jed_attack_gateway as gateway_defaults  # noqa: E402

BUDGET_S = gateway_defaults.DEFAULT_BUDGET_S
MAX_TOOL_HOPS = gateway_defaults.DEFAULT_MAX_TOOL_HOPS
ATTACK_SEED = gateway_defaults.ATTACK_SEED
ENV_SELECTION_NAME = gateway_defaults.ENV_SELECTION
MODEL_NAMES = list(gateway_defaults.MODEL_NAMES)
MODEL_PATHS = {"gpt_oss": str(GPT_OSS_PATH), "gemma": str(GEMMA_PATH)}
print(
    json.dumps(
        {
            "budget_s_per_model": BUDGET_S,
            "max_tool_hops": MAX_TOOL_HOPS,
            "attack_seed": ATTACK_SEED,
            "env_selection": ENV_SELECTION_NAME,
            "model_names": MODEL_NAMES,
        },
        indent=2,
    )
)

## Inject `attack.py`

The local runner replaces the next tagged cell with a base64-encoded copy of the selected attack.

In [ ]:
raise RuntimeError(
    "No attack was injected. Run this notebook through `jed-validate kaggle run path/to/attack.py`."
)

## Prepare the CUDA runtime

In [ ]:
if importlib.util.find_spec("llama_cpp") is None:
    print("Installing llama-cpp-python CUDA wheel...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "--no-cache-dir",
            "llama-cpp-python",
            "--extra-index-url",
            "https://abetlen.github.io/llama-cpp-python/whl/cu124",
        ],
        check=True,
    )
llama_cpp = importlib.import_module("llama_cpp")
print(f"llama-cpp-python ready: {llama_cpp.__file__}")

In [ ]:
from aicomp_sdk.core.env.api import EnvSelection
from aicomp_sdk.evaluation.diagnostics import EvaluatorVerbosity, RunDiagnostics
from aicomp_sdk.evaluation.runner import evaluate_redteam
from kaggle_evaluation.jed_attack_134815 import gemma_model_server, gpt_oss_model_server
from kaggle_evaluation.jed_attack_134815.gguf_model_server import GgufModelServer


def load_attack_class(path: Path):
    spec = importlib.util.spec_from_file_location("aas_submitted_attack", str(path))
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module.AttackAlgorithm


def build_gguf_agent_factory(model_name: str):
    if model_name == "gpt_oss":
        spec = gpt_oss_model_server.SPEC
    elif model_name == "gemma":
        spec = gemma_model_server.SPEC
    else:
        raise ValueError(f"Unknown model: {model_name}")
    server = GgufModelServer(spec)
    started = time.time()
    print(f"Loading {model_name} GGUF...")
    try:
        server.load_model()
    except Exception:
        unload_model_server(server, model_name)
        raise
    print(f"Loaded {model_name} in {time.time() - started:.1f}s")
    return (lambda: server._load_agent()), server


def unload_model_server(server, label: str):
    if server is not None:
        try:
            server.unload()
        except Exception as error:
            print(f"{label} unload error: {error!r}")
    gc.collect()
    try:
        import torch

        torch.cuda.empty_cache()
    except Exception:
        pass


def evaluate_model(AttackAlgorithm, model_name: str):
    transcript_file = ARTIFACTS_DIR / f"{model_name}_transcript.log"
    framework_file = ARTIFACTS_DIR / f"{model_name}_framework.jsonl"
    agent_debug_file = ARTIFACTS_DIR / f"{model_name}_agent-debug.jsonl"
    model_server = None
    started = time.time()
    try:
        agent_factory, model_server = build_gguf_agent_factory(model_name)
        with RunDiagnostics(
            EvaluatorVerbosity.DEBUG,
            transcript_file=transcript_file,
            event_log_file=framework_file,
            agent_debug_file=agent_debug_file,
        ) as diagnostics:
            execution = evaluate_redteam(
                AttackAlgorithm,
                budget_s=BUDGET_S,
                agent_factory=agent_factory,
                agent_label=f"{model_name}_gguf",
                env_selection=EnvSelection.GYM,
                fixtures_dir=COMP_DIR / "aicomp_sdk" / "fixtures",
                diagnostics=diagnostics,
                attack_env_seed=ATTACK_SEED,
            )
        attack = execution.attack
        summary = {
            "status": "complete",
            "model": model_name,
            "model_path": MODEL_PATHS[model_name],
            "score_normalized_0_to_1000": attack.score,
            "score_raw": attack.score_raw,
            "findings_count": attack.findings_count,
            "unique_cells": attack.unique_cells,
            "unique_canonical_cells": attack.unique_canonical_cells,
            "evaluation_time_s": attack.time_taken,
            "wall_time_s": time.time() - started,
            "guardrail_id": attack.guardrail_id,
            "guardrail_version": attack.guardrail_version,
            "env": str(execution.env_selection),
        }
        (ARTIFACTS_DIR / f"{model_name}_summary.json").write_text(
            json.dumps(summary, indent=2), encoding="utf-8"
        )
        (ARTIFACTS_DIR / f"{model_name}_findings.json").write_text(
            json.dumps(attack.findings, indent=2, default=str), encoding="utf-8"
        )
        print(json.dumps(summary, indent=2))
        if RUN_INTERACTIVE_INSPECTION:
            print("Interactive inspection is enabled; add experiments before model unload here.")
        return summary
    finally:
        unload_model_server(model_server, model_name)

## Run both authoritative model validations

In [ ]:
VALIDATION_STARTED_AT = datetime.now(timezone.utc)
VALIDATION_STARTED_MONOTONIC = time.time()
model_summaries = {}
failures = {}


def write_combined_summary():
    complete_scores = [
        item["score_normalized_0_to_1000"]
        for item in model_summaries.values()
        if item.get("status") == "complete"
    ]
    combined = {
        "status": (
            "failed"
            if failures
            else ("complete" if len(model_summaries) == len(MODEL_NAMES) else "running")
        ),
        "attack_sha256": ATTACK_SHA256,
        "started_at": VALIDATION_STARTED_AT.isoformat(),
        "completed_at": datetime.now(timezone.utc).isoformat(),
        "wall_time_s": time.time() - VALIDATION_STARTED_MONOTONIC,
        "budget_s_per_model": BUDGET_S,
        "max_tool_hops": MAX_TOOL_HOPS,
        "attack_seed": ATTACK_SEED,
        "env_selection": ENV_SELECTION_NAME,
        "model_paths": MODEL_PATHS,
        "models": model_summaries,
        "failures": failures,
        "local_public_mean": (
            sum(complete_scores) / len(MODEL_NAMES)
            if len(complete_scores) == len(MODEL_NAMES)
            else None
        ),
    }
    (ARTIFACTS_DIR / "validation_summary.json").write_text(
        json.dumps(combined, indent=2), encoding="utf-8"
    )
    return combined


try:
    AttackAlgorithm = load_attack_class(ATTACK_PATH)
except Exception as error:
    failures["attack_import"] = {
        "status": "failed",
        "error_type": type(error).__name__,
        "error": str(error),
        "traceback": traceback.format_exc(),
    }
    write_combined_summary()
    raise

for model_name in MODEL_NAMES:
    print(f"\n{'=' * 32} {model_name.upper()} {'=' * 32}")
    try:
        model_summaries[model_name] = evaluate_model(AttackAlgorithm, model_name)
    except Exception as error:
        failure = {
            "status": "failed",
            "model": model_name,
            "model_path": MODEL_PATHS[model_name],
            "error_type": type(error).__name__,
            "error": str(error),
            "traceback": traceback.format_exc(),
        }
        failures[model_name] = failure
        model_summaries[model_name] = failure
        (ARTIFACTS_DIR / f"{model_name}_summary.json").write_text(
            json.dumps(failure, indent=2), encoding="utf-8"
        )
        print(f"{model_name} failed: {error!r}")
    finally:
        write_combined_summary()

validation_summary = write_combined_summary()
print("\nFinal validation summary:")
print(json.dumps(validation_summary, indent=2))
if failures:
    raise RuntimeError(f"Validation failed for: {', '.join(failures)}")